In [5]:
from google.colab import files

# This opens a file picker — upload ml_service.py and your CSV
uploaded = files.upload()

Saving hervoice_ml_training_data.csv to hervoice_ml_training_data.csv


In [6]:
!pip install xgboost scikit-learn pandas numpy joblib
import subprocess
subprocess.run(["nvidia-smi"])  # should show your T4 GPU info

CompletedProcess(args=['nvidia-smi'], returncode=0)

In [7]:
!python ml_service.py hervoice_ml_training_data.csv

2026-06-06 06:28:14  INFO      __main__ — Starting HerVoice SafetyPredictor training pipeline…
2026-06-06 06:28:14  INFO      __main__ — Dataset  : hervoice_ml_training_data.csv
2026-06-06 06:28:14  INFO      __main__ — Grid search: False
2026-06-06 06:28:14  INFO      __main__ — Loading dataset: hervoice_ml_training_data.csv
2026-06-06 06:28:14  INFO      __main__ — Dataset shape: (20000, 8)
2026-06-06 06:28:14  INFO      __main__ — Columns found: ['device_uuid', 'lat', 'lng', 'grid_cell_id', 'safety_rating', 'time_context', 'created_at', 'tags']
2026-06-06 06:28:14  INFO      __main__ — Engineering features…
2026-06-06 06:28:15  INFO      __main__ — Feature matrix shape: (20000, 29)  (columns: 29)
2026-06-06 06:28:15  INFO      __main__ — Split: 16000 train / 4000 test
2026-06-06 06:28:15  INFO      __main__ — Training XGBRegressor (USE_GRID_SEARCH=False)…
[0]	validation_0-rmse:0.75555
[100]	validation_0-rmse:0.36609
[200]	validation_0-rmse:0.36699
[300]	validation_0-rmse:0.36940
[40

In [8]:
import importlib.util, sys

# Load the module
spec = importlib.util.spec_from_file_location("ml_service", "ml_service.py")
ml   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ml)

# Train
metrics = ml.train("hervoice_ml_training_data.csv")
print(metrics)

# Single prediction
score = ml.predict_safety(
    latitude=28.6139,
    longitude=77.2090,
    tags="dimly_lit,sparse_crowd",
    grid_cell_id="GRID_001",
    created_at="2024-06-15 22:30:00",
)
print("Safety score:", score)

[0]	validation_0-rmse:0.75555
[100]	validation_0-rmse:0.36609
[200]	validation_0-rmse:0.36699
[300]	validation_0-rmse:0.36940
[400]	validation_0-rmse:0.37118
[499]	validation_0-rmse:0.37366

═════════════════════════════════════════════
  Model Evaluation
═════════════════════════════════════════════
TRAIN R²   : 0.8537
TRAIN MAE  : 0.2206
TRAIN RMSE : 0.2943

TEST R²   : 0.7725
TEST MAE  : 0.2826
TEST RMSE : 0.3737
═════════════════════════════════════════════


────────────────────────────────────────
Top 20 Feature Importances
────────────────────────────────────────
   1. time_context                         0.22746  █████████████████████████████████████████████
   2. hour                                 0.14289  ████████████████████████████
   3. latitude                             0.05136  ██████████
   4. tag_very_safe                        0.04828  █████████
   5. tag_security_present                 0.04750  █████████
   6. tag_good_transport                   0.04625  █████

In [ ]:
import shutil
shutil.make_archive("resources", "zip", "resources")
files.download("resources.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>